In [ ]:
import sys, subprocess

packages = ["numpy", "pandas", "matplotlib", "deap"]
for pkg in packages:
    print("Instalando", pkg)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    except subprocess.CalledProcessError as e:
        print(f"Fallo instalando {pkg}: returncode={e.returncode}")
        # mostrar salida de pip para diagnóstico
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--verbose"])
        break


In [ ]:
# ============================================================
# CELDA 1 IMPORTACIONES Y FUNCIÓN DE CARGA
# ============================================================
import sys, subprocess, importlib
print("Python:", sys.version.split()[0])

for pkg in ("numpy", "pandas", "matplotlib", "deap"):
    try:
        m = importlib.import_module(pkg)
        print(f"Import OK: {pkg}  version:", getattr(m, "__version__", "n/a"))
    except Exception as e:
        print(f"Import FAILED: {pkg} ->", e)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
from deap import base, creator, tools, algorithms
from copy import deepcopy

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Librerías cargadas correctamente.")

In [1]:
# ============================================================
# CELL 2 — DATOS PREDICTIVOS + PARÁMETROS DE USUARIO
# ============================================================
import numpy as np
import pandas as pd
import os

HOURS = 8760   # horas por año
DAYS = 365
DT    = 1.0

# ============================================================
# 1) CARGAR LAS SERIES PREDICTIVAS
# ============================================================
def load_series(fname):
    """
    Carga una serie horaria desde un CSV.

    - Si el CSV tiene 1 columna: asume que son valores puros (sin cabecera).
    - Si el CSV tiene 2 columnas: asume que la primera es datetime y la segunda
      es el valor numérico. Salta la cabecera.
    - Valida que la longitud sea un múltiplo de 8760 (1 año, 5 años, etc.).

    Retorna el array y el número de años detectados.
    """
    if not os.path.exists(fname):
        raise FileNotFoundError(f"No encontrado: {fname}")

    with open(fname, 'r') as f:
        first_line = f.readline().strip()
        n_cols = len(first_line.split(','))

    if n_cols == 1:
        arr = np.loadtxt(fname, delimiter=',')
    else:
        arr = np.loadtxt(fname, delimiter=',', skiprows=1, usecols=(1,))

    if len(arr) % 8760 != 0:
        raise ValueError(
            f"{fname} debe tener un múltiplo de 8760 filas (1 año, 2 años...). "
            f"Tiene {len(arr)} filas."
        )

    n_years = len(arr) // 8760
    print(f"✅ {fname}: {len(arr)} filas ({n_years} año(s) detectado(s))")
    return arr

# Unidades de los CSV:
#   - load_8760.csv:  kW (demanda predicha por el ensamble)
#   - solar_8760.csv: factor de capacidad (0-1)
#   - wind_8760.csv:  km/h (velocidad del viento)
#
# NOTA: si en algún momento el CSV de demanda viene en W, se debe
# convertir a kW antes de usarlo. Aquí se asume kW por defecto.

load_pred  = load_series('load_8760.csv')    # kW
solar_pred = load_series('solar_8760.csv')   # factor de capacidad (0-1)
wind_pred  = load_series('wind_8760.csv')    # km/h

# Número de años detectados (a partir de la longitud)
n_years_load  = len(load_pred) // HOURS
n_years_solar = len(solar_pred) // HOURS
n_years_wind  = len(wind_pred) // HOURS

# Validación de consistencia entre series
if not (n_years_load == n_years_solar == n_years_wind):
    raise ValueError(
        f"Las series tienen distinto número de años: "
        f"load={n_years_load}, solar={n_years_solar}, wind={n_years_wind}"
    )

print(f"📅 Años detectados: {n_years_load}")

# ============================================================
# 2) CONSTRUCCIÓN DEL DATAFRAME SINTÉTICO
# ============================================================
# Unidades explícitas en los nombres de columna para evitar confusiones.
df12 = pd.DataFrame({
    'Rad_Solar_Wm2': solar_pred * 1000,   # factor de capacidad (0-1) a W/m² aprox
    'Temp_C':        26.0,                 # temperatura promedio (°C)
    'Wind_kmh':      wind_pred,            # km/h directamente
    'Demanda_kW':    load_pred,            # kW directamente (NO multiplicar por 1000)
})

df24 = df12.copy()

print(f"BD 12h: {df12.shape[0]} filas | BD 24h: {df24.shape[0]} filas")

# ============================================================
# 3) CONVERSIONES DE UNIDADES (explícitas)
# ============================================================
WIND_TO_MS     = 1.0 / 3.6      # km/h -> m/s
SOLAR_TO_KW_M2 = 1.0 / 1000.0   # W/m² -> kW/m²
# DEMAND_TO_KW: NO se aplica. La demanda ya está en kW.

# ============================================================
# 4) SERIES AMBIENTALES
# ============================================================
T_amb    = df12['Temp_C'].values.astype(float)                          # °C
V_hourly = df12['Wind_kmh'].values.astype(float) * WIND_TO_MS           # m/s
G_hourly = df12['Rad_Solar_Wm2'].values.astype(float) * SOLAR_TO_KW_M2  # kW/m²

# --- Escalar solar para que coincida con el promedio del paper (5.3 kWh/m²/día) ---
G_TARGET_DAILY = 5.3
G_current_daily = G_hourly.sum() / (n_years_load * 365)
G_scale = G_TARGET_DAILY / G_current_daily
G_hourly = G_hourly * G_scale

print(f"\n--- SOLAR SCALING ---")
print(f"Before scaling: {G_current_daily:.2f} kWh/m²/day")
print(f"After scaling:  {G_hourly.sum()/(n_years_load*365):.2f} kWh/m²/day  (factor={G_scale:.4f})")
print(f"Max irradiance: {G_hourly.max():.3f} kW/m²")

print("\n--- ENVIRONMENTAL SERIES ---")
print(f"Wind  : mean={V_hourly.mean():.2f} m/s   "
      f"min={V_hourly.min():.2f}   max={V_hourly.max():.2f}")
print(f"Solar : mean={G_hourly.mean():.3f} kW/m²  "
      f"max={G_hourly.max():.3f}  "
      f"daily_avg={G_hourly.sum()/(n_years_load*365):.2f} kWh/m²/day")
print(f"Temp  : mean={T_amb.mean():.1f} °C   "
      f"min={T_amb.min():.1f}   max={T_amb.max():.1f}")

# ============================================================
# 5) DEMANDA
# ============================================================
demand_raw = df12['Demanda_kW'].values.astype(float)   # ya está en kW

# Para el perfil diario, usar el último año (o el primero si hay varios)
# Aquí se toma el primer año como representativo
demand_first_year = demand_raw[:HOURS]
demand_daily = demand_first_year.reshape(365, 24)
profile_24h = demand_daily.mean(axis=0)
profile_24h_norm = profile_24h / profile_24h.mean()

print("\n--- DEMAND PROFILE (first year, normalized, mean = 1) ---")
print(f"Raw demand mean: {demand_raw.mean():.3f} kW  "
      f"({demand_raw.mean()*24:.1f} kWh/day raw)")
print("Hour :", "  ".join(f"{h:>5d}" for h in range(24)))
print("Prof :", "  ".join(f"{v:>5.2f}" for v in profile_24h_norm))

# Energías diarias objetivo (del paper)
E_DAY_CASE1 = 46.0
E_DAY_CASE2 = 102.0
print(f"\nTarget daily energies: Case 1 = {E_DAY_CASE1} kWh/day, "
      f"Case 2 = {E_DAY_CASE2} kWh/day")

# ============================================================
# 6) PARÁMETROS DE LOS COMPONENTES
# ============================================================
r = 0.08
CRF_LIFE = 20
CRF = r * (1 + r)**CRF_LIFE / ((1 + r)**CRF_LIFE - 1)

# PV panel
largo_pv, ancho_pv = 1762, 1134
A_pv = (largo_pv * ancho_pv) / 1e6
Pn_fv = 445
PR = 0.75
alpha_p = -0.004
Tcst, NOCT = 25, 48
Vida_util_fv = 20
G_STC = 1000.0
T_STC = 25.0
P_mod = Pn_fv * PR / 1000.0

# Wind turbine
Vel_ng, Vela, Vel_max = 12, 3, 25
Pn_ag = 330
P_wt_rated = Pn_ag / 1000.0
Vida_util_ag = 20
v_ci, v_r, v_co = Vela, Vel_ng, Vel_max

# Diesel
Pnom_Diesel = 3500
P_gen_rated = Pnom_Diesel / 1000.0
afd_diesel, bfd_diesel = 0.24, 0.86
EF = 2.6
Vida_util_gen = 5

# Battery
Pdesc = 0.8
Cap_bat = 2600
Vida_util_bat = 5
C_bat_unit = Cap_bat / 1000.0
SOC_min = 1 - Pdesc
SOC_max = 0.95
eta_ch = eta_dis = 0.95

# CAPEX/OPEX
CAPEX_pv, CAPEX_wt = 350.0, 3500.0
CAPEX_bat, CAPEX_gen = 900.0, 4000.0
OPEX_pv, OPEX_wt, OPEX_bat, OPEX_gen = 0.01, 0.02, 0.01, 0.05
LIFE_bat, LIFE_gen = Vida_util_bat, Vida_util_gen

# Emissions (kg CO2-eq/kWh)
EM_pv, EM_wt, EM_bat = 0.0225, 0.0080, 0.0280
EM_gen = EF * 0.28

# Job creation
jobs_per_MWp_local, jobs_per_MW_wind = 1.5, 0.8
jobs_per_MWh_diesel, job_years_per_MUSD = 0.014, 8.0

print("\n--- COMPONENT PARAMETERS ---")
print(f"Panel effective power: {P_mod:.5f} kW  ({P_mod*1000:.2f} W)")
print(f"Wind turbine rated:    {P_wt_rated:.3f} kW  ({Pn_ag} W)")
print(f"Diesel rated:          {P_gen_rated:.3f} kW  ({Pnom_Diesel} W)")
print(f"Battery unit:          {C_bat_unit:.3f} kWh")
print(f"SOC range:             [{SOC_min:.2f}, {SOC_max:.2f}]")
print(f"Discount rate:         {r*100:.1f}%")
print(f"CRF:                   {CRF:.4f}")
print(f"Diesel emission factor:{EM_gen:.3f} kg CO2/kWh")

✅ load_8760.csv: 8760 filas (1 año(s) detectado(s))
✅ solar_8760.csv: 8760 filas (1 año(s) detectado(s))
✅ wind_8760.csv: 8760 filas (1 año(s) detectado(s))
📅 Años detectados: 1
BD 12h: 8760 filas | BD 24h: 8760 filas

--- SOLAR SCALING ---
Before scaling: 4.43 kWh/m²/day
After scaling:  5.30 kWh/m²/day  (factor=1.1976)
Max irradiance: 0.864 kW/m²

--- ENVIRONMENTAL SERIES ---
Wind  : mean=3.29 m/s   min=0.10   max=9.15
Solar : mean=0.221 kW/m²  max=0.864  daily_avg=5.30 kWh/m²/day
Temp  : mean=26.0 °C   min=26.0   max=26.0

--- DEMAND PROFILE (first year, normalized, mean = 1) ---
Raw demand mean: 220.311 kW  (5287.5 kWh/day raw)
Hour :     0      1      2      3      4      5      6      7      8      9     10     11     12     13     14     15     16     17     18     19     20     21     22     23
Prof :  1.09   1.04   0.99   0.94   0.91   0.86   0.84   0.82   0.81   0.81   0.82   0.87   0.97   1.15   1.24   1.23   1.16   1.13   1.10   1.05   0.99   0.99   1.06   1.09

Target daily

In [2]:
# ============================================================
# CELL 3 — Models and Simulation
# ============================================================

def pv_power(N_pv, G, T_amb):
    T_cell = T_amb + (NOCT - 20) * (G * 1000.0) / 800.0
    temp_factor = 1.0 + alpha_p * (T_cell - Tcst)
    temp_factor = np.clip(temp_factor, 0.0, 1.2)
    P = N_pv * P_mod * (G * 1000.0 / G_STC) * temp_factor
    return np.maximum(P, 0)


def wind_power(N_wt, V):
    P = np.zeros_like(V, dtype=float)
    m1 = (V >= v_ci) & (V < v_r)
    P[m1] = P_wt_rated * (V[m1] - v_ci) / (v_r - v_ci)
    m2 = (V >= v_r) & (V <= v_co)
    P[m2] = P_wt_rated
    return N_wt * P


def build_load_series(E_day, use_24h):
    """Build hourly load [kW] for one year from normalized profile."""
    if use_24h:
        prof = profile_24h_norm.copy()
    else:
        mask = np.zeros(24, dtype=bool)
        mask[[0,1,2,3,4,5,18,19,20,21,22,23]] = True
        prof = profile_24h_norm * mask
        prof = prof / prof.sum() * 24
    hourly = (E_day / 24.0) * prof
    return np.tile(hourly, DAYS)


def simulate_system(N_pv, N_wt, N_bat, N_gen, use_24h=False):
    E_day = E_DAY_CASE2 if use_24h else E_DAY_CASE1
    load = build_load_series(E_day, use_24h)

    P_pv = pv_power(N_pv, G_hourly, T_amb)
    P_wt = wind_power(N_wt, V_hourly)
    P_ren = P_pv + P_wt

    C_bat_total = N_bat * C_bat_unit
    SOC = np.zeros(HOURS)
    SOC[0] = SOC_max * C_bat_total
    P_bat = np.zeros(HOURS)

    P_gen_series = np.zeros(HOURS)
    deficit = np.zeros(HOURS)

    for t in range(HOURS):
        net = P_ren[t] - load[t]
        soc_prev = SOC[t-1] if t > 0 else SOC[0]

        if net >= 0:
            room = SOC_max * C_bat_total - soc_prev
            charge = min(net, room / eta_ch)
            P_bat[t] = -charge
            SOC[t] = soc_prev + charge * eta_ch
        else:
            need = -net
            avail = soc_prev - SOC_min * C_bat_total
            discharge = min(need, avail * eta_dis)
            P_bat[t] = discharge
            SOC[t] = soc_prev - discharge / eta_dis
            residual = need - discharge

            if residual > 1e-6:
                Pg = min(residual, P_gen_rated)
                P_gen_series[t] = Pg
                residual -= Pg
            if residual > 1e-6:
                deficit[t] = residual

        if SOC[t] < SOC_min * C_bat_total:
            SOC[t] = SOC_min * C_bat_total

    LPSP = deficit.sum() / max(load.sum(), 1e-6)

    CAPEX_total = (N_pv*CAPEX_pv + N_wt*CAPEX_wt +
                   N_bat*CAPEX_bat + N_gen*CAPEX_gen)
    OPEX_annual = (N_pv*CAPEX_pv*OPEX_pv + N_wt*CAPEX_wt*OPEX_wt +
                   N_bat*CAPEX_bat*OPEX_bat + N_gen*CAPEX_gen*OPEX_gen)
    repl = (N_bat*CAPEX_bat/(1+r)**LIFE_bat +
            N_gen*CAPEX_gen/(1+r)**LIFE_gen)
    TAC = CAPEX_total * CRF + OPEX_annual + repl * CRF
    E_served = load.sum() - deficit.sum()
    LCOE = TAC / max(E_served, 1e-6)

    E_pv_year  = P_pv.sum() * DT
    E_wt_year  = P_wt.sum() * DT
    E_bat_year = np.abs(P_bat).sum() * DT
    E_gen_year = P_gen_series.sum() * DT
    LCA = (EM_pv*E_pv_year + EM_wt*E_wt_year +
           EM_bat*E_bat_year + EM_gen*E_gen_year)

    P_pv_MWp = N_pv * P_mod / 1000.0
    P_wt_MW  = N_wt * P_wt_rated / 1000.0
    E_gen_MWh = E_gen_year / 1000.0
    INV_MUSD = CAPEX_total / 1e6

    JC = (jobs_per_MWp_local * P_pv_MWp
          + jobs_per_MW_wind * P_wt_MW
          + jobs_per_MWh_diesel * E_gen_MWh
          + job_years_per_MUSD * INV_MUSD / CRF_LIFE)

    return LPSP, LCOE, LCA, JC

# --- Check if hour-5 spike is systematic or driven by a few days ---
daily_profiles = demand_raw.reshape(DAYS, 24)

print("\n--- HOURLY DEMAND STATISTICS ACROSS 365 DAYS ---")
print(f"{'Hour':>4} {'mean':>8} {'median':>8} {'p95':>8} {'max':>8}")
for h in range(24):
    col = daily_profiles[:, h]
    print(f"{h:>4} {col.mean():>8.3f} {np.median(col):>8.3f} "
          f"{np.percentile(col, 95):>8.3f} {col.max():>8.3f}")

# --- Sanity check with the original paper configurations ---
L1, C1, E1, J1 = simulate_system(21, 1, 13, 1, use_24h=False)
print(f"Paper config Case 1 (12h): LPSP={L1*100:.2f}%  LCOE={C1:.4f} USD/kWh  "
      f"LCA={E1:.1f} kgCO2/yr  JC={J1:.2f} jobs")

L2, C2, E2, J2 = simulate_system(75, 2, 35, 2, use_24h=True)
print(f"Paper config Case 2 (24h): LPSP={L2*100:.2f}%  LCOE={C2:.4f} USD/kWh  "
      f"LCA={E2:.1f} kgCO2/yr  JC={J2:.2f} jobs")


--- HOURLY DEMAND STATISTICS ACROSS 365 DAYS ---
Hour     mean   median      p95      max
   0  239.670  235.659  303.342  303.342
   1  228.829  223.882  301.459  301.459
   2  218.100  216.961  291.687  291.687
   3  207.647  207.071  278.039  278.039
   4  199.578  194.109  269.798  271.892
   5  190.503  188.079  266.416  270.923
   6  185.375  183.534  261.179  264.044
   7  180.719  179.923  255.173  259.520
   8  178.900  178.147  253.267  256.687
   9  177.811  179.452  249.629  257.607
  10  181.388  188.845  250.300  250.887
  11  191.903  201.624  262.853  262.853
  12  213.548  217.881  307.001  307.001
  13  254.015  225.878  403.673  403.673
  14  274.134  240.197  447.272  447.272
  15  272.035  240.243  431.301  431.301
  16  256.254  241.799  386.721  386.721
  17  249.092  242.269  372.979  372.979
  18  243.258  236.855  372.111  372.111
  19  232.251  235.027  345.030  345.030
  20  219.050  230.310  304.057  304.057
  21  218.072  236.605  269.877  269.877
  22  2

In [3]:
# ============================================================
# CELDA 1 (2ª PARTE) — IMPORTACIONES Y FUNCIÓN DE CARGA
# ============================================================
import random
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt
import os
import json
from datetime import datetime

def load_series(fname):
    if not os.path.exists(fname):
        raise FileNotFoundError(f"No encontrado: {fname}")

    with open(fname, 'r') as f:
        first_line = f.readline().strip()
        n_cols = len(first_line.split(','))

    if n_cols == 1:
        arr = np.loadtxt(fname, delimiter=',')
    else:
        arr = np.loadtxt(fname, delimiter=',', skiprows=1, usecols=(1,))

    if len(arr) % 8760 != 0:
        raise ValueError(
            f"{fname} debe tener un múltiplo de 8760 filas. "
            f"Tiene {len(arr)} filas."
        )

    n_years = len(arr) // 8760
    print(f"✅ {fname}: {len(arr)} filas ({n_years} año(s))")
    return arr

In [4]:
# ============================================================
# CELDA 2 (2ª PARTE) — CARGA DE LAS SERIES HORARIAS
# ============================================================
load  = load_series('load_8760.csv')    # kW
solar = load_series('solar_8760.csv')   # factor de capacidad (0-1)
wind  = load_series('wind_8760.csv')    # km/h
H = len(load)

# Validación de consistencia
if not (len(load) == len(solar) == len(wind)):
    raise ValueError(
        f"Las series tienen distinta longitud: "
        f"load={len(load)}, solar={len(solar)}, wind={len(wind)}"
    )

print(f"✅ load: {len(load)} filas")
print(f"✅ solar: {len(solar)} filas")
print(f"✅ wind: {len(wind)} filas")
print(f"H = {H}  ({H // 8760} año(s))")

✅ load_8760.csv: 8760 filas (1 año(s))
✅ solar_8760.csv: 8760 filas (1 año(s))
✅ wind_8760.csv: 8760 filas (1 año(s))
✅ load: 8760 filas
✅ solar: 8760 filas
✅ wind: 8760 filas
H = 8760  (1 año(s))


In [5]:
# Celda 3: parámetros del modelo (edítalos con los valores exactos de tu tesis)
PARAMS = {
    "capex_pv": 600.0,            # CAPEX PV en USD/kW
    "capex_wind": 1200.0,         # CAPEX eólica en USD/kW
    "capex_batt": 400.0,          # CAPEX baterías en USD/kWh
    "capex_diesel": 200.0,        # CAPEX generador en USD/kW
    "om_frac": 0.02,              # fracción anual O&M
    "lifespan": 20,               # horizonte de vida útil en años
    "discount": 0.08,             # tasa de descuento anual (fracción)
    "embodied_pv_tco2_per_kw": 0.05,   # emisiones embebidas PV (tCO2e/kW instalado)
    "embodied_wind_tco2_per_kw": 0.04, # emisiones embebidas eólica (tCO2e/kW)
    "embodied_batt_tco2_per_kwh": 0.10,# emisiones embebidas baterías (tCO2e/kWh)
    "embodied_diesel_tco2_per_kw": 0.02,# emisiones embebidas generador (tCO2e/kW)
    "diesel_tco2_per_kwh": 0.27,       # emisiones operativas diésel (tCO2e/kWh)
    "diesel_kwh_per_liter": 3.0,       # rendimiento diésel (kWh por litro)
    "diesel_cost_per_liter": 1.0,      # precio diésel (USD/litro)
    "jobs_pv_per_kw": 0.006,           # empleos por kW PV (ejemplo)
    "jobs_wind_per_kw": 0.004,         # empleos por kW eólica
    "jobs_batt_per_kwh": 0.001,        # empleos por kWh batería
    "jobs_diesel_per_kw": 0.0005,      # empleos por kW generador
    "area_max_m2": 10000.0,             # área disponible total en m2
    "m2_per_kw_pv": 8.0,               # m2 por kW de PV
    "budget_max": 2000000.0,            # presupuesto máximo en USD
    "batt_roundtrip_eff": 0.90,        # eficiencia ida-vuelta de la batería (fracción)
    "batt_c_rate_max": 1.0,            # C-rate máximo (1C por defecto)
    "soc_min_frac": 0.20,              # SOC mínimo (fracción)
    "soc_max_frac": 0.95,              # SOC máximo (fracción)
    "pv_kw_max": 500.0,                # límite superior búsqueda PV (kW)
    "wind_kw_max": 500.0,              # límite superior búsqueda eólica (kW)
    "batt_kwh_max": 2000.0,            # límite superior búsqueda baterías (kWh)
    "diesel_kw_max": 500.0,            # límite superior búsqueda diésel (kW)
    "pop_size": 8,                    # tamaño de población (ajustar si CPU limitado)
    "n_gen": 5,                      # número de generaciones
    "seed": 42                         # semilla aleatoria para reproducibilidad
}


In [6]:
# Celda 4: desempaquetar parámetros en variables locales y definir anualización
capex_pv = PARAMS["capex_pv"]                     # CAPEX PV
capex_wind = PARAMS["capex_wind"]                 # CAPEX eólica
capex_batt = PARAMS["capex_batt"]                 # CAPEX baterías
capex_diesel = PARAMS["capex_diesel"]             # CAPEX diésel
om_frac = PARAMS["om_frac"]                       # O&M fracción
lifespan = PARAMS["lifespan"]                     # vida útil años
discount = PARAMS["discount"]                     # tasa descuento

embodied_pv_tco2_per_kw = PARAMS["embodied_pv_tco2_per_kw"]   # emisiones embebidas PV
embodied_wind_tco2_per_kw = PARAMS["embodied_wind_tco2_per_kw"] # emisiones embebidas eólica
embodied_batt_tco2_per_kwh = PARAMS["embodied_batt_tco2_per_kwh"] # emisiones embebidas baterías
embodied_diesel_tco2_per_kw = PARAMS["embodied_diesel_tco2_per_kw"] # emisiones embebidas diésel
diesel_tco2_per_kwh = PARAMS["diesel_tco2_per_kwh"]             # emisiones operativas diésel

diesel_kwh_per_liter = PARAMS["diesel_kwh_per_liter"]           # rendimiento diésel
diesel_cost_per_liter = PARAMS["diesel_cost_per_liter"]         # precio diésel

jobs_pv_per_kw = PARAMS["jobs_pv_per_kw"]                       # empleos PV
jobs_wind_per_kw = PARAMS["jobs_wind_per_kw"]                   # empleos eólica
jobs_batt_per_kwh = PARAMS["jobs_batt_per_kwh"]                 # empleos baterías
jobs_diesel_per_kw = PARAMS["jobs_diesel_per_kw"]               # empleos diésel

area_max_m2 = PARAMS["area_max_m2"]                             # área máxima disponible
m2_per_kw_pv = PARAMS["m2_per_kw_pv"]                           # m2 por kW PV
budget_max = PARAMS["budget_max"]                               # presupuesto máximo

batt_roundtrip_eff = PARAMS["batt_roundtrip_eff"]               # eficiencia ida-vuelta batería
batt_ch_eff = np.sqrt(batt_roundtrip_eff)                       # eficiencia de carga (raíz)
batt_dis_eff = np.sqrt(batt_roundtrip_eff)                      # eficiencia de descarga (raíz)
batt_c_rate_max = PARAMS["batt_c_rate_max"]                     # C-rate máximo
soc_min_frac = PARAMS["soc_min_frac"]                           # SOC mínimo fracción
soc_max_frac = PARAMS["soc_max_frac"]                           # SOC máximo fracción

pv_kw_max = PARAMS["pv_kw_max"]                                 # límite PV
wind_kw_max = PARAMS["wind_kw_max"]                             # límite eólica
batt_kwh_max = PARAMS["batt_kwh_max"]                           # límite baterías
diesel_kw_max = PARAMS["diesel_kw_max"]                         # límite diésel

POP_SIZE = PARAMS["pop_size"]                                   # tamaño población
N_GEN = PARAMS["n_gen"]                                         # generaciones
SEED = PARAMS["seed"]                                           # semilla

# función para anualizar CAPEX (anualización tipo annuity)
def annuity(capex, life=lifespan, r=discount):
    if r == 0:                                                  # si r==0 evitar división por cero
        return capex / life                                     # coste anual simple
    return capex * (r * (1 + r) ** life) / ((1 + r) ** life - 1) # factor anualización

# anualizar CAPEX unitarios para uso posterior
ann_pv = annuity(capex_pv)
ann_wind = annuity(capex_wind)
ann_batt = annuity(capex_batt)
ann_diesel = annuity(capex_diesel)


In [7]:
# Celda 5: función que evalúa un individuo (PV_kW, Wind_kW, Batt_kWh, Diesel_kW)
def evaluate_individual(ind):
    pv_kw, wind_kw, batt_kwh, diesel_kw = ind                    # desempaquetar variables de decisión
    gen_pv = pv_kw * solar                                       # generación PV horaria (kW)
    gen_wind = wind_kw * wind                                    # generación eólica horaria (kW)
    gen_total = gen_pv + gen_wind                                # generación renovable total por hora

    soc = soc_max_frac * batt_kwh                                # SOC inicial (fracción alta para simulación)
    soc_min = soc_min_frac * batt_kwh                            # SOC mínimo absoluto (kWh)
    soc_max = soc_max_frac * batt_kwh                            # SOC máximo absoluto (kWh)
    batt_max_charge_power = batt_kwh * batt_c_rate_max           # potencia máxima de carga (kW)
    batt_max_discharge_power = batt_kwh * batt_c_rate_max        # potencia máxima de descarga (kW)

    unmet = 0.0                                                   # energía no satisfecha acumulada (kWh)
    diesel_energy_kwh = 0.0                                       # energía suministrada por diésel (kWh)
    energy_served = 0.0                                           # energía total servida (kWh)

    for t in range(H):                                            # bucle horario (8760 horas)
        supply = gen_total[t]                                     # generación renovable en la hora t
        demand = load[t]                                          # demanda en la hora t
        if supply >= demand:                                      # si la generación cubre la demanda
            energy_served += demand                               # toda la demanda servida
            surplus = supply - demand                             # excedente disponible para carga
            max_charge = min(batt_max_charge_power, soc_max - soc) # límite de carga por potencia y espacio SOC
            charge_possible = min(surplus, max_charge / batt_ch_eff) # energía disponible para cargar (considerando eficiencia)
            charge = charge_possible * batt_ch_eff                # energía realmente almacenada (kWh)
            soc += charge                                         # actualizar SOC
        else:
            deficit = demand - supply                             # déficit que debe cubrirse
            max_discharge = min(batt_max_discharge_power, soc - soc_min) # límite de descarga por potencia y SOC
            discharge_possible = min(deficit / batt_dis_eff, max_discharge) # energía que se puede descargar
            discharge = discharge_possible
            soc -= discharge                                      # reducir SOC por descarga
            supplied_by_batt = discharge * batt_dis_eff           # energía efectiva suministrada por batería
            remaining_deficit = deficit - supplied_by_batt       # déficit restante tras batería
            if remaining_deficit > 1e-9:
                diesel_supply = min(remaining_deficit, diesel_kw) # diésel suministra hasta su potencia nominal
                diesel_energy_kwh += diesel_supply               # acumular energía diésel
                supplied_total = supply + supplied_by_batt + diesel_supply # total suministrado esa hora
                energy_served += min(supplied_total, demand)     # sumar energía servida (no exceder demanda)
                rem2 = remaining_deficit - diesel_supply
                if rem2 > 1e-6:
                    unmet += rem2                                # si aún queda déficit, contabilizar como unmet
            else:
                energy_served += demand                           # batería cubrió el déficit, demanda servida

    total_demand = load.sum()                                     # demanda anual total (kWh)

    # Costos anuales: anualización CAPEX + O&M + combustible diésel
    annualized_capex = ann_pv * pv_kw + ann_wind * wind_kw + ann_batt * batt_kwh + ann_diesel * diesel_kw
    annual_om = om_frac * (capex_pv*pv_kw + capex_wind*wind_kw + capex_batt*batt_kwh + capex_diesel*diesel_kw)
    diesel_liters = diesel_energy_kwh / diesel_kwh_per_liter
    diesel_fuel_cost = diesel_liters * diesel_cost_per_liter
    annual_cost = annualized_capex + annual_om + diesel_fuel_cost

    # Objetivos principales
    lcoe = annual_cost / (energy_served + 1e-9)                    # LCOE USD/kWh (evitar división por cero)
    lpsp = unmet / (total_demand + 1e-9)                          # LPSP fracción de demanda no satisfecha

    # LCA: emisiones embebidas anualizadas + emisiones operativas diésel
    embodied_ann_pv = embodied_pv_tco2_per_kw * pv_kw / lifespan
    embodied_ann_wind = embodied_wind_tco2_per_kw * wind_kw / lifespan
    embodied_ann_batt = embodied_batt_tco2_per_kwh * batt_kwh / lifespan
    embodied_ann_diesel = embodied_diesel_tco2_per_kw * diesel_kw / lifespan
    operational_diesel_emissions = diesel_energy_kwh * diesel_tco2_per_kwh
    lca = embodied_ann_pv + embodied_ann_wind + embodied_ann_batt + embodied_ann_diesel + operational_diesel_emissions

    # JC: empleos estimados según factores por unidad instalada
    jc = jobs_pv_per_kw * pv_kw + jobs_wind_per_kw * wind_kw + jobs_batt_per_kwh * batt_kwh + jobs_diesel_per_kw * diesel_kw

    # Penalizaciones por violación de restricciones (área y presupuesto) y por soluciones no servibles
    area_used = pv_kw * m2_per_kw_pv
    capex_total = capex_pv*pv_kw + capex_wind*wind_kw + capex_batt*batt_kwh + capex_diesel*diesel_kw
    penalty = 0.0
    PENALTY_FACTOR = 1e6
    if area_used > area_max_m2:
        penalty += PENALTY_FACTOR * (area_used - area_max_m2) / max(area_max_m2,1.0)  # penalizar exceso de área
    if capex_total > budget_max:
        penalty += PENALTY_FACTOR * (capex_total - budget_max) / max(budget_max,1.0)    # penalizar exceso de presupuesto
    if energy_served < 0.05 * total_demand:
        penalty += PENALTY_FACTOR                                                       # penalizar soluciones que sirven muy poca energía

    # Retornar tupla de objetivos: minimizamos LCOE, LPSP, LCA; max JC -> minimizamos -JC
    return (lcoe + penalty, lpsp + penalty, lca + penalty, -jc + penalty)


In [8]:
# Celda 6: configuración de DEAP (crear tipos, operadores y población inicial)
random.seed(SEED)                                              # fijar semilla para reproducibilidad
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0, -1.0, -1.0))  # crear fitness multiobjetivo (minimizar 4 objetivos)
creator.create("Individual", list, fitness=creator.FitnessMulti)               # crear tipo Individual como lista con fitness multi

toolbox = base.Toolbox()                                        # crear toolbox para registrar operadores
# registrar generadores de atributos (variables continuas con rango)
toolbox.register("attr_pv", random.uniform, 0.0, pv_kw_max)     # PV_kW aleatorio entre 0 y pv_kw_max
toolbox.register("attr_wind", random.uniform, 0.0, wind_kw_max) # Wind_kW aleatorio
toolbox.register("attr_batt", random.uniform, 0.0, batt_kwh_max)# Batt_kWh aleatorio
toolbox.register("attr_diesel", random.uniform, 0.0, diesel_kw_max) # Diesel_kW aleatorio

# registrar inicializador de individuo y población
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_pv, toolbox.attr_wind, toolbox.attr_batt, toolbox.attr_diesel), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# registrar funciones de evaluación, cruce, mutación y selección
toolbox.register("evaluate", evaluate_individual)               # función de evaluación definida antes
toolbox.register("mate", tools.cxSimulatedBinaryBounded, low=[0,pv_kw_max,0,0], up=[pv_kw_max,wind_kw_max,batt_kwh_max,diesel_kw_max], eta=15.0)
toolbox.register("mutate", tools.mutPolynomialBounded, low=[0,0,0,0], up=[pv_kw_max,wind_kw_max,batt_kwh_max,diesel_kw_max], eta=20.0, indpb=0.2)
toolbox.register("select", tools.selNSGA2)                      # selección NSGA-II


In [9]:
# Celda 7: ejecutar el NSGA-II con DEAP (bucle evolutivo)
def run_nsga2():
    pop = toolbox.population(n=POP_SIZE)

    fitnesses = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

    CXPB = 0.9
    MUTPB = 0.2

    for gen in range(1, N_GEN + 1):
        offspring = tools.selTournament(pop, len(pop), tournsize=3)
        offspring = [toolbox.clone(ind) for ind in offspring]

        for i in range(0, len(offspring), 2):
            if random.random() <= CXPB and i + 1 < len(offspring):
                toolbox.mate(offspring[i], offspring[i + 1])
                del offspring[i].fitness.values
                del offspring[i + 1].fitness.values

        for mutant in offspring:
            if random.random() <= MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        pop = toolbox.select(pop + offspring, POP_SIZE)

        if gen % 10 == 0 or gen == 1 or gen == N_GEN:
            print(f"Generación {gen} completada")

    return pop                 # imprimir progreso cada 10 generaciones


In [10]:
def main():
    # --- Ejecutar el bucle evolutivo ---
    pop = run_nsga2()

    # --- Extraer Frente de Pareto ---
    pareto = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]
    rows = []
    for ind in pareto:
        pv_kw, wind_kw, batt_kwh, diesel_kw = ind
        lcoe, lpsp, lca, neg_jc = ind.fitness.values
        rows.append([
            pv_kw, wind_kw, batt_kwh, diesel_kw,
            lcoe, lpsp, lca, -neg_jc,
            pv_kw * m2_per_kw_pv,
            capex_pv * pv_kw + capex_wind * wind_kw + capex_batt * batt_kwh + capex_diesel * diesel_kw,
        ])

    df = pd.DataFrame(rows, columns=[
        'PV_kW', 'Wind_kW', 'Batt_kWh', 'Diesel_kW',
        'LCOE', 'LPSP', 'LCA_tCO2a', 'JC_jobs',
        'Area_m2', 'Capex_USD',
    ])
    df.to_csv('pareto_deap.csv', index=False)
    print(f"Pareto guardado en pareto_deap.csv, soluciones: {len(df)}")

    # --- Guardar metadatos ---
    metadata = {
        "fecha": datetime.now().isoformat(),
        "pop_size": POP_SIZE,
        "n_gen": N_GEN,
        "seed": SEED,
        "n_years": len(load) // 8760,
        "configuracion": {
            "capex_pv": capex_pv,
            "capex_wind": capex_wind,
            "capex_batt": capex_batt,
            "capex_diesel": capex_diesel,
            "budget_max": budget_max,
            "area_max_m2": area_max_m2,
        },
        "n_soluciones": len(df),
    }
    with open('pareto_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    print("Metadatos guardados en pareto_metadata.json")

    # --- Gráfico ---
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(df['LCOE'], df['LPSP'], c=df['LCA_tCO2a'], cmap='plasma', s=30)
    plt.colorbar(sc, label='LCA (tCO2e/a)')
    plt.xlabel('LCOE (USD/kWh)')
    plt.ylabel('LPSP (fracción)')
    plt.title('Frente Pareto (DEAP)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('pareto_deap_lcoe_lpsp.png', dpi=200)
    plt.close()
    print("Gráfico guardado: pareto_deap_lcoe_lpsp.png")


if __name__ == "__main__":
    main()

Generación 1 completada
Generación 5 completada
Pareto guardado en pareto_deap.csv, soluciones: 8
Metadatos guardados en pareto_metadata.json
Gráfico guardado: pareto_deap_lcoe_lpsp.png
